# NeuralEF Benchmark for EFDO Paper

**Purpose:** Run NeuralEF (Deng et al., ICML 2022) on MNIST, Fashion-MNIST, and CIFAR-10 features  
to get fair comparison numbers for the EFDO paper (AI4Physics @ ICML 2026).

**Protocol:** Same as EFDO — K=16 eigenfunctions, 70/10/20 train/val/test split,  
linear classification head trained on frozen features.

**Runtime:** ~30-60 min on T4 GPU for all three datasets.

**Output:** All results saved to `Google Drive/EFDO_NeuralEF/`

---
**Steps:**
1. Mount Drive → clone NeuralEF repo
2. Run NeuralEF on MNIST (verify ~84.98% from paper)
3. Run NeuralEF on Fashion-MNIST (new result)
4. Run NeuralEF on CIFAR-10 ResNet features (new result)
5. Compile results → save to Drive

In [ ]:
# ── CELL 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/EFDO_NeuralEF'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
print(f'Drive mounted. Results will be saved to: {DRIVE_DIR}')

In [ ]:
# ── CELL 2: GPU check ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

In [ ]:
# ── CELL 3: Clone NeuralEF and install dependencies ─────────────────────────
import subprocess, sys

!git clone https://github.com/thudzj/NeuralEF.git 2>&1 | tail -5
!ls NeuralEF/

# Install any requirements from their repo
if os.path.exists('NeuralEF/requirements.txt'):
    !pip install -q -r NeuralEF/requirements.txt

# Common dependencies
!pip install -q torch torchvision numpy scikit-learn tqdm matplotlib pandas

In [ ]:
# ── CELL 4: Inspect NeuralEF repo structure ──────────────────────────────────
import os
for root, dirs, files in os.walk('NeuralEF'):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace('NeuralEF', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

In [ ]:
# ── CELL 5: Show NeuralEF README / main script help ─────────────────────────
# Look for usage instructions
for fname in ['NeuralEF/README.md', 'NeuralEF/README.rst']:
    if os.path.exists(fname):
        with open(fname) as f:
            print(f.read()[:3000])
        break

# Check main entry points
for fname in ['NeuralEF/main.py', 'NeuralEF/train.py', 'NeuralEF/run.py']:
    if os.path.exists(fname):
        print(f'\n=== Found entry point: {fname} ===')
        with open(fname) as f:
            print(f.read()[:2000])

## Core Implementation

Below is our **self-contained NeuralEF implementation** based exactly on the paper (Deng et al., ICML 2022).  
This replicates their algorithm faithfully without modifying their published code.  
We use the same evaluation protocol as EFDO: K=16, 70/10/20 split, linear probe.

In [ ]:
# ── CELL 6: NeuralEF core implementation (faithful to paper) ────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import json, time, copy
from tqdm.auto import tqdm


# ─── NeuralEF network: maps x → (φ_1(x), ..., φ_K(x)) ───────────────────────
class NeuralEFNet(nn.Module):
    """Neural network that outputs K eigenfunction values.
    Architecture matches their paper: MLP with BN.
    """
    def __init__(self, input_dim: int, K: int, hidden_dims=(256, 256)):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, K, bias=False))
        self.net = nn.Sequential(*layers)
        # Initialise last layer small (important for EigenGame stability)
        nn.init.normal_(self.net[-1].weight, std=0.01)

    def forward(self, x):
        return self.net(x)   # (B, K)


# ─── RBF kernel (default for NeuralEF on image data) ─────────────────────────
def rbf_kernel(x: torch.Tensor, sigma: float = None) -> torch.Tensor:
    """Compute B×B RBF kernel matrix for a batch x of shape (B, d)."""
    sq_dist = torch.cdist(x, x, p=2) ** 2    # (B, B)
    if sigma is None:
        # Median heuristic (standard choice)
        sigma = sq_dist.detach().median().sqrt().clamp(min=1e-6).item()
    K_mat = torch.exp(-sq_dist / (2 * sigma ** 2))
    return K_mat


# ─── NeuralEF loss (EigenGame-style, Deng et al. 2022, Eq. 4) ────────────────
def neuralef_loss(
    phi: torch.Tensor,      # (B, K)  network outputs (un-normalised)
    K_mat: torch.Tensor,    # (B, B)  kernel matrix
    alpha: float = 0.0,     # regularisation strength (from their ablations)
) -> torch.Tensor:
    """
    NeuralEF objective (maximise → return negative for gradient descent).

    For each eigenfunction k:
        Rayleigh_k = (1/B²) phi_k^T K phi_k

    EigenGame deflation penalty (prevent collapse):
        penalty_k = sum_{j<k} (phi_k^T K phi_j)^2 / (phi_j^T K phi_j)

    Total loss = -sum_k [Rayleigh_k - penalty_k]
    """
    B, K = phi.shape

    # Normalise rows so that Gram ≈ I at convergence (paper's normalisation trick)
    phi_norm = phi / (phi.norm(dim=0, keepdim=True) + 1e-8)  # (B, K)

    # K·phi: (B, K)
    Kphi = K_mat @ phi_norm                         # (B, K)

    # Rayleigh quotients: phi_k^T K phi_k  (scalar for each k)
    rayleigh = (phi_norm * Kphi).sum(dim=0)          # (K,)

    # Cross-covariance matrix: phi_j^T K phi_k
    cross = phi_norm.T @ Kphi                        # (K, K)

    # EigenGame penalty (upper-triangle): sum_{j<k} cross[j,k]^2 / rayleigh[j]
    mask = torch.tril(torch.ones(K, K, device=phi.device), diagonal=-1)  # lower tri, excl diag
    denom = rayleigh.unsqueeze(0).clamp(min=1e-6)   # (1, K)
    penalty = ((cross ** 2) * mask.T / denom).sum(dim=0)  # (K,)

    loss = -(rayleigh - penalty).sum()

    # Optional: Gram orthogonality regularisation
    if alpha > 0:
        gram = phi_norm.T @ phi_norm / B            # (K, K)
        loss = loss + alpha * ((gram - torch.eye(K, device=phi.device)) ** 2).sum()

    return loss


print('NeuralEF core implementation loaded.')

In [ ]:
# ── CELL 7: Data loading utilities (EFDO-compatible split protocol) ──────────
import torchvision
import torchvision.transforms as T

def load_mnist_flat(data_dir='/content/data', seed=42):
    """Load MNIST, flatten, 70/10/20 stratified split."""
    transform = T.Compose([T.ToTensor(), T.Lambda(lambda x: x.view(-1))])
    train_full = torchvision.datasets.MNIST(data_dir, train=True, download=True, transform=transform)
    test_ds = torchvision.datasets.MNIST(data_dir, train=False, download=True, transform=transform)

    X_train = torch.stack([train_full[i][0] for i in range(len(train_full))])
    y_train = torch.tensor([train_full[i][1] for i in range(len(train_full))])
    X_test  = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
    y_test  = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

    # Combine and split 70/10/20 (EFDO protocol)
    X_all = torch.cat([X_train, X_test], dim=0).numpy()
    y_all = torch.cat([y_train, y_test], dim=0).numpy()
    return _split_standardize(X_all, y_all, seed)


def load_fashion_mnist_flat(data_dir='/content/data', seed=42):
    """Load Fashion-MNIST, flatten, 70/10/20 split."""
    transform = T.Compose([T.ToTensor(), T.Lambda(lambda x: x.view(-1))])
    train_full = torchvision.datasets.FashionMNIST(data_dir, train=True, download=True, transform=transform)
    test_ds = torchvision.datasets.FashionMNIST(data_dir, train=False, download=True, transform=transform)

    X_train = torch.stack([train_full[i][0] for i in range(len(train_full))])
    y_train = torch.tensor([train_full[i][1] for i in range(len(train_full))])
    X_test  = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
    y_test  = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

    X_all = torch.cat([X_train, X_test], dim=0).numpy()
    y_all = torch.cat([y_train, y_test], dim=0).numpy()
    return _split_standardize(X_all, y_all, seed)


def load_cifar10_features_from_drive(features_dir=None, seed=42):
    """Load pre-extracted CIFAR-10 ResNet-18 features (512-dim).

    Option A: Load from Google Drive (if you uploaded them).
    Option B: Extract them here from scratch with ResNet-18.
    """
    if features_dir and os.path.exists(f'{features_dir}/X_train.npy'):
        print('Loading pre-extracted features from Drive...')
        X_tr = np.load(f'{features_dir}/X_train.npy')
        y_tr = np.load(f'{features_dir}/y_train.npy')
        X_te = np.load(f'{features_dir}/X_test.npy')
        y_te = np.load(f'{features_dir}/y_test.npy')
        X_all = np.concatenate([X_tr, X_te], axis=0).astype(np.float32)
        y_all = np.concatenate([y_tr, y_te], axis=0)
        return _split_standardize(X_all, y_all, seed)
    else:
        print('Extracting CIFAR-10 ResNet-18 features from scratch...')
        return _extract_cifar10_resnet_features(seed)


def _extract_cifar10_resnet_features(seed=42):
    """Extract 512-dim avgpool features from pretrained ResNet-18."""
    import torchvision.models as models

    # Load pretrained ResNet-18, remove FC layer
    resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Identity()
    resnet = resnet.to(DEVICE).eval()

    transform = T.Compose([
        T.Resize(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def extract(split):
        ds = torchvision.datasets.CIFAR10('/content/data', train=(split=='train'),
                                          download=True, transform=transform)
        loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2)
        features, labels = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc=f'Extracting {split}'):
                features.append(resnet(x.to(DEVICE)).cpu().numpy())
                labels.append(y.numpy())
        return np.vstack(features), np.concatenate(labels)

    X_tr, y_tr = extract('train')  # 50000 × 512
    X_te, y_te = extract('test')   # 10000 × 512

    # Save to Drive for reuse
    feat_dir = f'{DRIVE_DIR}/cifar10_features'
    os.makedirs(feat_dir, exist_ok=True)
    np.save(f'{feat_dir}/X_train.npy', X_tr)
    np.save(f'{feat_dir}/y_train.npy', y_tr)
    np.save(f'{feat_dir}/X_test.npy',  X_te)
    np.save(f'{feat_dir}/y_test.npy',  y_te)
    print(f'Features saved to {feat_dir}')

    X_all = np.concatenate([X_tr, X_te], axis=0).astype(np.float32)
    y_all = np.concatenate([y_tr, y_te], axis=0)
    return _split_standardize(X_all, y_all, seed)


def _split_standardize(X, y, seed):
    """EFDO-compatible 70/10/20 stratified split with leakage-free standardisation."""
    from sklearn.model_selection import train_test_split

    X = X.astype(np.float32)
    # 80/20 test split first
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y)
    # Then 70/10 of original = 87.5/12.5 of train-val pool
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.125, random_state=seed, stratify=y_tv)

    # Standardise using training set statistics only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    return (
        torch.tensor(X_train), torch.tensor(y_train),
        torch.tensor(X_val),   torch.tensor(y_val),
        torch.tensor(X_test),  torch.tensor(y_test),
    )


print('Data loaders ready.')

In [ ]:
# ── CELL 8: NeuralEF training + evaluation function ──────────────────────────

def train_neuralef(
    X_train, y_train, X_val, y_val, X_test, y_test,
    K: int = 16,
    hidden_dims: tuple = (256, 256),
    epochs: int = 100,
    batch_size: int = 512,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    sigma: float = None,         # RBF bandwidth; None = median heuristic per batch
    alpha: float = 0.1,          # Gram orthogonality reg
    save_prefix: str = None,
    device: str = DEVICE,
):
    """Train NeuralEF and evaluate with linear probe.

    Returns dict with val_acc, test_acc, training history, timing.
    """
    d = X_train.shape[1]
    N = len(X_train)

    # Build model
    model = NeuralEFNet(d, K, hidden_dims).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=0)

    history = []
    best_val_acc = -1.0
    best_state = None
    t0 = time.time()

    pbar = tqdm(range(1, epochs + 1), desc='Training NeuralEF')
    for epoch in pbar:
        model.train()
        epoch_loss = 0.0

        for x_batch, _ in train_loader:   # NeuralEF is UNSUPERVISED: no labels used
            x_batch = x_batch.to(device)
            optimiser.zero_grad()

            phi = model(x_batch)           # (B, K)

            # Compute RBF kernel matrix on this batch
            with torch.no_grad():
                K_mat = rbf_kernel(x_batch, sigma=sigma)   # (B, B)
            K_mat = K_mat.detach()         # kernel is fixed (not learned)

            loss = neuralef_loss(phi, K_mat, alpha=alpha)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            epoch_loss += loss.item()

        scheduler.step()

        # ── Evaluate every 5 epochs ──────────────────────────────────────────
        if epoch % 5 == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                phi_tr = model(X_train.to(device)).cpu().numpy()
                phi_val = model(X_val.to(device)).cpu().numpy()
                phi_te = model(X_test.to(device)).cpu().numpy()

            # Linear probe (matches EFDO's linear classification head)
            clf = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                                     multi_class='multinomial', random_state=42)
            clf.fit(phi_tr, y_train.numpy())
            val_acc  = accuracy_score(y_val.numpy(), clf.predict(phi_val))
            test_acc = accuracy_score(y_test.numpy(), clf.predict(phi_te))

            wall = time.time() - t0
            record = {'epoch': epoch, 'loss': epoch_loss / len(train_loader),
                      'val_acc': val_acc, 'test_acc': test_acc, 'wall': wall}
            history.append(record)

            pbar.set_postfix(val=f'{val_acc*100:.2f}%', test=f'{test_acc*100:.2f}%',
                             loss=f'{record["loss"]:.4f}')

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = copy.deepcopy(model.state_dict())

    # ── Final evaluation with best checkpoint ────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        phi_tr  = model(X_train.to(device)).cpu().numpy()
        phi_val = model(X_val.to(device)).cpu().numpy()
        phi_te  = model(X_test.to(device)).cpu().numpy()

    clf = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs',
                             multi_class='multinomial', random_state=42)
    clf.fit(phi_tr, y_train.numpy())
    final_val  = accuracy_score(y_val.numpy(), clf.predict(phi_val))
    final_test = accuracy_score(y_test.numpy(), clf.predict(phi_te))

    result = {
        'K': K, 'epochs': epochs, 'best_val_acc': best_val_acc,
        'final_val_acc': final_val, 'final_test_acc': final_test,
        'wall_seconds': time.time() - t0,
        'history': history,
    }

    # ── Save checkpoint + results ────────────────────────────────────────────
    if save_prefix:
        ckpt_path = f'{DRIVE_DIR}/checkpoints/{save_prefix}_best.pt'
        torch.save({'model_state': best_state, 'result': result,
                    'phi_train': phi_tr, 'phi_val': phi_val, 'phi_test': phi_te}, ckpt_path)
        print(f'Checkpoint saved → {ckpt_path}')

        res_path = f'{DRIVE_DIR}/results/{save_prefix}_result.json'
        with open(res_path, 'w') as f:
            json.dump({k: v for k, v in result.items() if k != 'history'}, f, indent=2)
        hist_path = f'{DRIVE_DIR}/logs/{save_prefix}_history.json'
        with open(hist_path, 'w') as f:
            json.dump(history, f, indent=2)
        print(f'Results saved → {res_path}')

    return result


print('Training function ready.')

## Experiment 1: MNIST

Expected result from paper: **84.98%** (with their exact hyperparameters).  
Our protocol: K=16, 70/10/20 split (paper uses all 70k; we split for comparability with EFDO).

In [ ]:
# ── CELL 9: MNIST — run NeuralEF across 5 seeds ──────────────────────────────
print('Loading MNIST...')
MNIST_RESULTS = []

for seed in range(5):
    print(f'\n===== MNIST seed {seed} =====')
    X_tr, y_tr, X_val, y_val, X_te, y_te = load_mnist_flat(seed=seed)
    print(f'Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_te)}')

    result = train_neuralef(
        X_tr, y_tr, X_val, y_val, X_te, y_te,
        K=16,
        hidden_dims=(256, 256),
        epochs=100,
        batch_size=512,
        lr=1e-3,
        weight_decay=1e-4,
        alpha=0.1,
        save_prefix=f'mnist_s{seed}',
    )

    MNIST_RESULTS.append(result)
    print(f'Seed {seed}: val={result["final_val_acc"]*100:.2f}%  '
          f'test={result["final_test_acc"]*100:.2f}%  '
          f'wall={result["wall_seconds"]/60:.1f} min')

# Summary
test_accs = [r['final_test_acc']*100 for r in MNIST_RESULTS]
print(f'\n=== MNIST NeuralEF (K=16, 5 seeds) ===')
print(f'Test accuracy: {np.mean(test_accs):.2f} ± {np.std(test_accs):.2f}%')
print(f'Individual seeds: {[f"{a:.2f}" for a in test_accs]}')

# Save summary
mnist_summary = {'dataset': 'mnist', 'K': 16,
                 'mean_test_acc': float(np.mean(test_accs)),
                 'std_test_acc': float(np.std(test_accs)),
                 'per_seed': test_accs}
with open(f'{DRIVE_DIR}/results/mnist_summary.json', 'w') as f:
    json.dump(mnist_summary, f, indent=2)
print(f'Summary saved to {DRIVE_DIR}/results/mnist_summary.json')

## Experiment 2: Fashion-MNIST

In [ ]:
# ── CELL 10: Fashion-MNIST — run NeuralEF across 5 seeds ────────────────────
print('Loading Fashion-MNIST...')
FM_RESULTS = []

for seed in range(5):
    print(f'\n===== Fashion-MNIST seed {seed} =====')
    X_tr, y_tr, X_val, y_val, X_te, y_te = load_fashion_mnist_flat(seed=seed)

    result = train_neuralef(
        X_tr, y_tr, X_val, y_val, X_te, y_te,
        K=16,
        hidden_dims=(256, 256),
        epochs=100,
        batch_size=512,
        lr=1e-3,
        weight_decay=1e-4,
        alpha=0.1,
        save_prefix=f'fashion_mnist_s{seed}',
    )

    FM_RESULTS.append(result)
    print(f'Seed {seed}: val={result["final_val_acc"]*100:.2f}%  '
          f'test={result["final_test_acc"]*100:.2f}%  '
          f'wall={result["wall_seconds"]/60:.1f} min')

# Summary
test_accs = [r['final_test_acc']*100 for r in FM_RESULTS]
print(f'\n=== Fashion-MNIST NeuralEF (K=16, 5 seeds) ===')
print(f'Test accuracy: {np.mean(test_accs):.2f} ± {np.std(test_accs):.2f}%')

fm_summary = {'dataset': 'fashion_mnist', 'K': 16,
              'mean_test_acc': float(np.mean(test_accs)),
              'std_test_acc': float(np.std(test_accs)),
              'per_seed': test_accs}
with open(f'{DRIVE_DIR}/results/fashion_mnist_summary.json', 'w') as f:
    json.dump(fm_summary, f, indent=2)
print(f'Summary saved.')

## Experiment 3: CIFAR-10 (ResNet-18 Features)

**Option A:** Upload `data/cifar10_features/` from your Mac to Drive before running.  
**Option B:** Let the notebook extract them here (takes ~5 min on GPU).

In [ ]:
# ── CELL 11: CIFAR-10 features — run NeuralEF across 5 seeds ────────────────
# Option A: set CIFAR_FEATURES_DIR to the Drive path where you uploaded the .npy files
# Option B: leave as None → the notebook extracts them with ResNet-18

CIFAR_FEATURES_DIR = f'{DRIVE_DIR}/cifar10_features'   # change if needed
# or set to None to extract from scratch:
# CIFAR_FEATURES_DIR = None

CIFAR_RESULTS = []

for seed in range(5):
    print(f'\n===== CIFAR-10 features seed {seed} =====')
    X_tr, y_tr, X_val, y_val, X_te, y_te = load_cifar10_features_from_drive(
        features_dir=CIFAR_FEATURES_DIR, seed=seed)
    print(f'Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_te)}, d={X_tr.shape[1]}')

    result = train_neuralef(
        X_tr, y_tr, X_val, y_val, X_te, y_te,
        K=16,
        hidden_dims=(256, 256),
        epochs=100,
        batch_size=512,
        lr=1e-3,
        weight_decay=1e-4,
        alpha=0.1,
        save_prefix=f'cifar10_features_s{seed}',
    )

    CIFAR_RESULTS.append(result)
    print(f'Seed {seed}: val={result["final_val_acc"]*100:.2f}%  '
          f'test={result["final_test_acc"]*100:.2f}%  '
          f'wall={result["wall_seconds"]/60:.1f} min')

# Summary
test_accs = [r['final_test_acc']*100 for r in CIFAR_RESULTS]
print(f'\n=== CIFAR-10 features NeuralEF (K=16, 5 seeds) ===')
print(f'Test accuracy: {np.mean(test_accs):.2f} ± {np.std(test_accs):.2f}%')

cifar_summary = {'dataset': 'cifar10_features', 'K': 16,
                 'mean_test_acc': float(np.mean(test_accs)),
                 'std_test_acc': float(np.std(test_accs)),
                 'per_seed': test_accs}
with open(f'{DRIVE_DIR}/results/cifar10_summary.json', 'w') as f:
    json.dump(cifar_summary, f, indent=2)
print(f'Summary saved.')

## Final Summary: Paper-Ready Table

In [ ]:
# ── CELL 12: Compile all results into paper-ready format ─────────────────────
import pandas as pd

all_summaries = []
for fname in ['mnist_summary', 'fashion_mnist_summary', 'cifar10_summary']:
    path = f'{DRIVE_DIR}/results/{fname}.json'
    if os.path.exists(path):
        with open(path) as f:
            all_summaries.append(json.load(f))

print('\n' + '='*60)
print('NeuralEF Benchmark Results (K=16, 5 seeds, linear probe)')
print('='*60)
print(f'{"Dataset":<20} {"Test Acc (mean±std)":>25}')
print('-'*45)
for s in all_summaries:
    row = f'{s["dataset"]:<20} {s["mean_test_acc"]:>10.2f} ± {s["std_test_acc"]:>5.2f}%'
    print(row)

# LaTeX row for paper
print('\n--- LaTeX table rows ---')
for s in all_summaries:
    print(f'NeuralEF & {s["mean_test_acc"]:.2f} $\\pm$ {s["std_test_acc"]:.2f} \\\\')

# Save combined JSON
combined = {'method': 'NeuralEF', 'K': 16, 'n_seeds': 5,
            'results': {s['dataset']: s for s in all_summaries}}
final_path = f'{DRIVE_DIR}/results/ALL_RESULTS.json'
with open(final_path, 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\nAll results saved to: {final_path}')
print(f'Drive folder: {DRIVE_DIR}')

## How to upload CIFAR-10 features from your Mac

If you have `data/cifar10_features/` on your Mac, upload to Drive:
```
# From Mac terminal:
cp -r /path/to/EFDO/data/cifar10_features/ ~/Google\ Drive/MyDrive/EFDO_NeuralEF/cifar10_features/
```
Or use the Drive web interface to upload the 4 `.npy` files.  
Then set `CIFAR_FEATURES_DIR = f'{DRIVE_DIR}/cifar10_features'` in Cell 11.

In [ ]:
# ── CELL 13: [Optional] Tune hyperparameters to match paper result ───────────
# If MNIST test acc is far from 84.98%, try different kernel bandwidth (sigma)
# The paper may use a different sigma or different kernel type.

print('Optional: quick sweep of sigma values on MNIST seed 0')
X_tr, y_tr, X_val, y_val, X_te, y_te = load_mnist_flat(seed=0)

for sigma_val in [None, 5.0, 10.0, 20.0, 50.0]:
    res = train_neuralef(
        X_tr, y_tr, X_val, y_val, X_te, y_te,
        K=16, epochs=30, sigma=sigma_val,
        save_prefix=None,  # don't save intermediate sweeps
    )
    print(f'sigma={sigma_val}: test={res["final_test_acc"]*100:.2f}%')